# AVG Extraction

Jalankan setelah `AVG Training.ipynb` selesai. Notebook ini memuat CNN tersimpan, mengekstrak embedding dari 50.000 gambar training resmi, lalu menyimpan fitur, label, indeks, split, dan hash model ke `train_features.npz`. Arsip ini digunakan oleh Feature Engineering Combination.

## 1. Konfigurasi
Tetapkan run yang sama dan siapkan folder fitur.

In [1]:
from pathlib import Path
import hashlib
import time

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RUN_ID = "cifar10_7fitur_v1"
SEED = 30092026
VARIANT = "avg"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / VARIANT

for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
if gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Perangkat GPU:", gpus, "| Run:", RUN_ID)

I0000 00:00:1791470147.516098  175391 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791470147.552924  175391 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791470148.505225  175391 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Perangkat GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')] | Run: cifar10_7fitur_v1


## 2. Data dan split
Gunakan bagian training resmi saja dan hitung kembali split identik.

In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
assert x_all.shape == (50000, 32, 32, 3)

train_idx, val_idx = train_test_split(
    np.arange(len(y_all)),
    test_size=0.2,
    stratify=y_all,
    random_state=SEED,
)
train_idx = np.sort(train_idx)
val_idx = np.sort(val_idx)
assert len(train_idx) == 40000
assert len(val_idx) == 10000

split_hash = hashlib.sha256(
    train_idx.tobytes() + val_idx.tobytes()
).hexdigest()
print("Split:", len(train_idx), len(val_idx), split_hash[:16])

Split: 40000 10000 16004a1ac8a89ac9


## 3. Konversi citra
Gunakan rumus kanal sama persis dengan Training.

In [3]:
def prepare_images(x, variant):
    x = np.asarray(x, dtype=np.float32) / 255.0

    if variant == "rgb":
        return x
    if variant == "avg":
        return np.mean(x, axis=-1, keepdims=True, dtype=np.float32)
    if variant == "ntsc":
        weights = np.array([0.299, 0.587, 0.114], dtype=np.float32)
        return np.sum(x * weights, axis=-1, keepdims=True)

    raise ValueError(f"Varian tidak dikenal: {variant}")

## 4. Muat model CNN
Embedding diambil dari layer bernama `embedding` pada checkpoint terbaik.

In [4]:
model_path = OUT / "model" / "cnn.keras"
assert model_path.exists(), "Jalankan notebook Training lebih dahulu"
model_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()

cnn = tf.keras.models.load_model(model_path, compile=False)
extractor = tf.keras.Model(
    inputs=cnn.input,
    outputs=cnn.get_layer("embedding").output,
)
assert extractor.output_shape[-1] == 512

I0000 00:00:1791470151.245695  175391 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1952 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


## 5. Ekstraksi embedding
Proses dalam batch agar penggunaan memori terkendali.

In [5]:
start = time.perf_counter()
features = extractor.predict(
    prepare_images(x_all, VARIANT),
    batch_size=128,
    verbose=1,
).astype(np.float32)

assert features.shape == (50000, 512)
assert np.isfinite(features).all()
print("Durasi ekstraksi tersimpan (detik):", time.perf_counter() - start)

I0000 00:00:1791470152.849621  175461 cuda_dnn.cc:461] Loaded cuDNN version 92600


391/391 ━━━━━━━━━━━━━━━━━━━━ 22s 54ms/step
Durasi ekstraksi tersimpan (detik): 23.040704682003707


## 6. Simpan fitur training
Arsip memuat indeks, label, identitas split, dan hash model untuk menolak artefak yang tidak cocok.

In [6]:
feature_path = OUT / "fitur" / "train_features.npz"
np.savez_compressed(
    feature_path,
    features=features,
    y=y_all,
    indices=np.arange(len(y_all)),
    train_idx=train_idx,
    val_idx=val_idx,
    run_id=RUN_ID,
    variant=VARIANT,
    split_hash=split_hash,
    model_sha=model_sha,
    preprocessing_version="v1",
)
print("Fitur tersimpan:", feature_path)

Fitur tersimpan: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v1/avg/fitur/train_features.npz
